# TIF Econometría II: El Niño Costero y la Bolsa de Valores de Lima (BVL)
## Módulo de Extracción, Integración y Transformación de Datos (2015 - 2026)

Este notebook implementa el pipeline integral de recolección de datos diseñado en la guía metodológica del TIF (*El Niño Costero y la BVL.pdf*).

### 1. Variables procedentes de la API del BCRP:
- **Índice General BVL**: `PN01142MM`
- **Índice Selectivo BVL**: `PN01143MM`
- **Índices Sectoriales BVL (S&P/BVL)**:
  - Financiero (*SP/BVL Financial*): `PN01148MM`
  - Industrial (*SP/BVL Industrial*): `PN01149MM`
  - Minería (*SP/BVL Mining*): `PN01150MM`
  - Servicios (*SP/BVL Services*): `PN01151MM`
- **Variables Macroeconómicas y Financieras de Control**:
  - Tipo de Cambio PEN/USD (promedio): `PN01234PM`
  - Tasa de Referencia de Política Monetaria: `PD04722MM`
  - Cobre LME (¢US$/lb, promedio): `PN01652XM`
  - Petróleo WTI (US$/barril, promedio): `PN01660XM`
  - Oro LME (US$/oz tr, promedio): `PN01654XM`
  - Actividad Económica (PBI mensual, índice 2007=100): `PN01770AM`
- **Expectativas Macroeconómicas y Empresariales (Encuesta Mensual BCRP)**:
  - Expectativa de Inflación a 12 meses: `PD12912AM`
  - Expectativa de PBI a 12 meses: `PD38048AM`
  - Expectativa de Tipo de Cambio a 12 meses: `PD38049AM`
  - Expectativa de la Economía a 3 meses (Confianza Empresarial): `PD38045AM`
  - Expectativa del Sector a 3 meses: `PD38046AM`

### 2. Variables que NO están en el BCRP (obtenidas de fuentes externas oficiales):
1. **ICEN (Índice Costero El Niño)**:
   - **Fuente:** Instituto Geofísico del Perú (IGP) / ENFEN.
   - **URL:** `http://met.igp.gob.pe/datos/ICEN.txt`
   - **Naturaleza:** Variable explicativa principal (media móvil de 3 meses de anomalías de TSM en región Niño 1+2, oficial de Perú).
2. **S&P 500 (`^GSPC`)**:
   - **Fuente:** Yahoo Finance.
   - **Naturaleza:** Co-movimiento global de mercados accionarios.
3. **VIX (`^VIX`)**:
   - **Fuente:** Chicago Board Options Exchange (CBOE) vía Yahoo Finance.
   - **Naturaleza:** Proxy de incertidumbre y aversión al riesgo internacional.

In [ ]:
import io
import json
import requests
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

# Configuración de visualización
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Librerías importadas exitosamente.")

### 3. Extracción de Series desde la API del BCRP
Se utiliza el endpoint oficial del BCRP:
`https://estadisticas.bcrp.gob.pe/estadisticas/series/api/[codigo]/json/[periodo_ini]/[periodo_fin]/esp`

Incluimos tanto las series financieras/reales como las expectativas de la encuesta mensual del BCRP.

In [ ]:
# Diccionario de códigos BCRP a descargar
BCRP_SERIES = {
    # Series bursátiles BVL
    'bvl_general': 'PN01142MM',        # S&P/BVL Peru General
    'bvl_selectivo': 'PN01143MM',      # S&P/BVL Peru Select
    'bvl_financiero': 'PN01148MM',     # SP/BVL Financial
    'bvl_industrial': 'PN01149MM',     # SP/BVL Industrial
    'bvl_mineria': 'PN01150MM',        # SP/BVL Mining
    'bvl_servicios': 'PN01151MM',      # SP/BVL Services
    
    # Variables macro y precios internacionales
    'tc_pen_usd': 'PN01234PM',         # Tipo de cambio interbancario promedio (S/ por USD)
    'tasa_ref': 'PD04722MM',           # Tasa de referencia de política monetaria (%)
    'cobre_lme': 'PN01652XM',          # Cobre LME (centavos US$/lb)
    'petroleo_wti': 'PN01660XM',       # Petróleo WTI (US$/barril)
    'oro_lme': 'PN01654XM',            # Oro LME (US$/oz tr)
    'pbi_indice': 'PN01770AM',          # Índice mensual de PBI (INEI/BCRP, 2007=100)
    
    # Expectativas macroeconómicas y empresariales (Encuesta de Expectativas BCRP)
    'exp_inflacion_12m': 'PD12912AM',  # Expectativa de Inflación a 12 meses (%)
    'exp_pbi_12m': 'PD38048AM',        # Expectativa de PBI a 12 meses (%)
    'exp_tc_12m': 'PD38049AM',         # Expectativa de TC a 12 meses (S/ por USD)
    'exp_emp_economia_3m': 'PD38045AM',# Confianza empresarial: expectativas economía a 3m
    'exp_emp_sector_3m': 'PD38046AM'   # Confianza empresarial: expectativas sector a 3m
}

START_PERIOD = "2015-1"
END_PERIOD = "2026-5"

MESES_ES = {
    'Ene': 1, 'Feb': 2, 'Mar': 3, 'Abr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Ago': 8, 'Set': 9, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dic': 12
}

def parse_bcrp_period(period_str):
    """Convierte cadenas como 'Ene.2015' a pd.Period con frecuencia mensual."""
    parts = period_str.replace(' ', '').split('.')
    if len(parts) == 2:
        m_str, y_str = parts[0], parts[1]
        month = MESES_ES.get(m_str, 1)
        year = int(y_str)
        return pd.Period(year=year, month=month, freq='M')
    return None

print("Descargando series desde la API del BCRP...")
series_frames = []

for var_name, code in BCRP_SERIES.items():
    url = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{code}/json/{START_PERIOD}/{END_PERIOD}/esp"
    try:
        resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
        data = resp.json()
        periods = data.get('periods', [])
        
        idx = [parse_bcrp_period(p['name']) for p in periods]
        vals = []
        for p in periods:
            raw_val = p['values'][0]
            try:
                vals.append(float(raw_val) if raw_val not in ('n.d.', '', None) else np.nan)
            except ValueError:
                vals.append(np.nan)
        
        s = pd.Series(vals, index=pd.PeriodIndex(idx, freq='M'), name=var_name)
        series_frames.append(s)
        print(f"  [OK] {code} -> {var_name:20} ({len(s)} obs | Último: {vals[-1]})")
    except Exception as e:
        print(f"  [ERROR] {code} -> {var_name}: {e}")

df_bcrp = pd.concat(series_frames, axis=1).sort_index()
print(f"\nDataset BCRP consolidado: {df_bcrp.shape[0]} filas, {df_bcrp.shape[1]} columnas.")
df_bcrp.tail()

### 4. Extracción de Datos Externos (NO disponibles en BCRP)

#### 4.1 Índice Costero El Niño (ICEN) - IGP / ENFEN
El ICEN mide las anomalías de la Temperatura Superficial del Mar (TSM) en la región Niño 1+2 frente a las costas peruanas con una media móvil trimestral. Es la variable climática oficial peruana.

In [ ]:
print("Descargando ICEN desde met.igp.gob.pe...")
url_icen = "http://met.igp.gob.pe/datos/ICEN.txt"
resp_icen = requests.get(url_icen, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)

icen_rows = []
for line in resp_icen.text.splitlines():
    line = line.strip()
    if not line or line.startswith('%'):
        continue
    parts = line.split()
    if len(parts) >= 3:
        year = int(parts[0])
        month = int(parts[1])
        val = float(parts[2])
        icen_rows.append({
            'period': pd.Period(year=year, month=month, freq='M'),
            'icen': val
        })

df_icen = pd.DataFrame(icen_rows).set_index('period')
# Filtramos para la muestra del proyecto (desde 2015)
df_icen = df_icen.loc['2015-01':'2026-05']
print(f"ICEN descargado: {len(df_icen)} meses (Rango: {df_icen.index.min()} a {df_icen.index.max()})")
df_icen.tail()

#### 4.2 S&P 500 y VIX - Yahoo Finance
Descargamos los índices internacionales para controlar por co-movimiento y apetito por riesgo global.

In [ ]:
print("Descargando S&P 500 (^GSPC) y VIX (^VIX) desde Yahoo Finance...")
yf_raw = yf.download(['^GSPC', '^VIX'], start='2015-01-01', end='2026-06-01', interval='1mo', progress=False)['Close']

# Convertir índice de fecha a periodo mensual
yf_df = yf_raw.rename(columns={'^GSPC': 'sp500', '^VIX': 'vix'}).copy()
yf_df.index = pd.PeriodIndex(yf_df.index, freq='M')
yf_df = yf_df.loc['2015-01':'2026-05']
print(f"Datos internacionales descargados: {len(yf_df)} observaciones.")
yf_df.tail()

### 5. Integración del Dataset Maestro y Transformaciones Econométricas
Siguiendo la tabla de operacionalización del documento:
- **Retornos continuos (log-retornos)**: $r_{i,t} = 100 \times \ln(P_{i,t} / P_{i,t-1})$ para BVL, commodities y S&P 500.
- **Primeras diferencias de tasas y expectativas**:
  - $\Delta i_t = i_t - i_{t-1}$ (tasa de referencia)
  - $\Delta Exp\_PBI_t$, $\Delta Exp\_Inf_t$, $\Delta Exp\_TC_t$
  - $\Delta Exp\_Empresarial_t$
- **Variables climáticas derivadas**:
  - $ICEN_t$: Nivel continuo de anomalía térmica (°C).
  - $ICEN\_calido_t = \max(ICEN_t, 0)$: Para medir asimetría en shocks cálidos (El Niño).
  - $ICEN\_frio_t = \min(ICEN_t, 0)$: Para shocks fríos (La Niña).
  - Dummy Niño Costero: 1 si $ICEN_t \ge 1.0$ (condición de evento Niño según ENFEN).

In [ ]:
# Fusión de todas las fuentes en un DataFrame maestro alineado
df_master = df_bcrp.join([df_icen, yf_df], how='outer').sort_index()

# Construcción de Retornos Logarítmicos (r = 100 * ln(P_t / P_{t-1}))
price_vars = [
    'bvl_general', 'bvl_selectivo', 'bvl_financiero', 'bvl_industrial', 
    'bvl_mineria', 'bvl_servicios', 'tc_pen_usd', 'cobre_lme', 
    'petroleo_wti', 'oro_lme', 'pbi_indice', 'sp500'
]

for col in price_vars:
    df_master[f'ret_{col}'] = 100 * (np.log(df_master[col]) - np.log(df_master[col].shift(1)))

# Diferencias de tasa de referencia y expectativas macro/empresariales
diff_vars = [
    'tasa_ref', 'exp_inflacion_12m', 'exp_pbi_12m', 'exp_tc_12m', 
    'exp_emp_economia_3m', 'exp_emp_sector_3m'
]
for var in diff_vars:
    df_master[f'd_{var}'] = df_master[var] - df_master[var].shift(1)

# Descomposición del ICEN para pruebas de asimetría
df_master['icen_calido'] = df_master['icen'].apply(lambda x: max(x, 0.0) if pd.notnull(x) else np.nan)
df_master['icen_frio'] = df_master['icen'].apply(lambda x: min(x, 0.0) if pd.notnull(x) else np.nan)
df_master['dummy_nino'] = (df_master['icen'] >= 1.0).astype(int)

# Dummy COVID-19 (marzo 2020 a diciembre 2020) para análisis de quiebre/robustez
df_master['dummy_covid'] = ((df_master.index >= '2020-03') & (df_master.index <= '2020-12')).astype(int)

print("Estructura del Dataset Maestro:")
print(f"Dimensiones: {df_master.shape}")
print(f"Rango temporal: {df_master.index.min()} a {df_master.index.max()}")
df_master[['icen', 'ret_bvl_general', 'exp_pbi_12m', 'd_exp_pbi_12m', 'exp_emp_economia_3m', 'd_exp_emp_economia_3m']].tail(10)

### 6. Estadísticas Descriptivas y Visualización

In [ ]:
print("=== ESTADÍSTICAS DESCRIPTIVAS DE RETORNOS, EXPECTATIVAS E ICEN ===")
cols_summary = [
    'icen', 'ret_bvl_general', 'ret_bvl_financiero', 'ret_bvl_mineria', 
    'exp_pbi_12m', 'exp_emp_economia_3m', 'ret_cobre_lme', 'sp500'
]
summary_table = df_master[cols_summary].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
summary_table.columns = ['N', 'Media', 'Desv. Est.', 'Mínimo', 'Mediana', 'Máximo']
display(summary_table.round(3))

# Gráfico conjunto: ICEN vs. Retorno BVL y Expectativas Empresariales
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), dpi=100, sharex=True)
dates = df_master.index.to_timestamp()

# Panel 1: ICEN vs Retorno BVL
color_icen = '#d95f02'
ax1.set_title('A. Fenómeno El Niño (ICEN) y Retornos del Índice General BVL (2015-2026)', fontsize=12, fontweight='bold')
ax1.set_ylabel('ICEN (°C)', color=color_icen, fontsize=10)
ax1.plot(dates, df_master['icen'], color=color_icen, lw=2.2, label='ICEN (°C)')
ax1.axhline(0, color='gray', linestyle='--', alpha=0.6)
ax1.axhline(1.0, color='red', linestyle=':', alpha=0.7, label='Umbral Niño (+1.0°C)')
ax1.axhline(-1.0, color='blue', linestyle=':', alpha=0.7, label='Umbral Niña (-1.0°C)')
ax1.tick_params(axis='y', labelcolor=color_icen)
ax1.legend(loc='upper left')

ax1_b = ax1.twinx()
color_bvl = '#1b9e77'
ax1_b.set_ylabel('Retorno BVL General (%)', color=color_bvl, fontsize=10)
ax1_b.bar(dates, df_master['ret_bvl_general'], color=color_bvl, alpha=0.35, width=20, label='Retorno BVL (%)')
ax1_b.tick_params(axis='y', labelcolor=color_bvl)

# Panel 2: ICEN vs Expectativas de PBI y Confianza Empresarial
ax2.set_title('B. Canal de Transmisión: Expectativas de PBI (12m) y Confianza Empresarial (3m)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Año', fontsize=10)
ax2.set_ylabel('Expectativa PBI 12m (%)', color='#2b5c8f', fontsize=10)
ax2.plot(dates, df_master['exp_pbi_12m'], color='#2b5c8f', lw=2.0, label='Expectativa PBI 12m (%)')
ax2.tick_params(axis='y', labelcolor='#2b5c8f')
ax2.legend(loc='upper left')

ax2_b = ax2.twinx()
ax2_b.set_ylabel('Confianza Empresarial (Puntos)', color='#7570b3', fontsize=10)
ax2_b.plot(dates, df_master['exp_emp_economia_3m'], color='#7570b3', lw=1.8, linestyle='--', label='Confianza Economía 3m')
ax2_b.axhline(50, color='gray', linestyle=':', alpha=0.5, label='Umbral Neutro (50)')
ax2_b.tick_params(axis='y', labelcolor='#7570b3')
ax2_b.legend(loc='upper right')

fig.tight_layout()
plt.show()

### 7. Exportación del Dataset para Estimaciones en Stata y Python
Exportamos el dataset maestro en dos formatos:
- `datos_bvl_icen_master.csv`: Para uso en Python / R.
- `datos_bvl_icen_master.dta`: Base lista para Stata con formato de fecha temporal `%tm` (`tsset stata_tm`).

In [ ]:
import os

# Crear copia con columnas de tiempo formateadas
df_export = df_master.copy()
df_export['year'] = df_export.index.year
df_export['month'] = df_export.index.month
df_export['date_str'] = df_export.index.strftime('%Y-%m')

# Stata mensual: meses desde 1960m1
df_export['stata_tm'] = (df_export['year'] - 1960) * 12 + (df_export['month'] - 1)

# Exportar CSV
csv_path = 'datos_bvl_icen_master.csv'
df_export.to_csv(csv_path, index=False)
print(f"Dataset exportado a CSV: {csv_path}")

# Exportar DTA para Stata
dta_path = 'datos_bvl_icen_master.dta'
try:
    df_export.to_stata(dta_path, write_index=False, version=118)
    print(f"Dataset exportado a Stata (.dta): {dta_path}")
except Exception as e:
    print(f"Aviso al exportar a Stata: {e}")

print("\n¡Proceso de recolección e integración finalizado con éxito!")